# Chapter 4:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//modern-recommender-systems/blob/main/notebooks/chapter-01/example_notebook.ipynb)

This notebook introduces popular item lists.

In [2]:
# Cell 2: Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu


ModuleNotFoundError: No module named 'recsys'

In [ ]:

# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.data import loaders
from recsys.models import CollaborativeFiltering

DATA_PATH = get_data_path()

In [ ]:
print("Ready to build recommender systems!")

In [ ]:
import numpy as np
import pandas as pd

# Simulation parameters
n_intervals = 10                    # Number of time intervals (e.g. 10 ten-minute buckets)
n_users_per_interval = 200          # User visits per interval
items_per_interval = 5              # Number of candidate items
top_K = 3                           # For candidate set reduction (optional)

np.random.seed(42)

# 1. Generate ground-truth response rates (P_jt) for each item and interval
# Here, randomly sample response rates for illustration
P_jt = pd.DataFrame(
    np.random.uniform(0.05, 0.25, size=(n_intervals, items_per_interval)),
    columns=[f'item_{j}' for j in range(items_per_interval)]
)
P_jt.head(10)

In [ ]:
# 3. Simulate the environment and the most-popular model

overall_clicks = 0
overall_recs = 0

for t in range(n_intervals):
    # Candidates and response rates for this interval

    candidates = P_jt.columns.tolist()

    pjs = P_jt.iloc[t].to_dict()
    n_visits = n_users_per_interval


    # --- Decide x_j for each item (serving fractions) using popularity up to t-1 ---
    if t == 0:
        # No data: Uniform serving fraction
        x_j = {j: 1/len(candidates) for j in candidates}
    else:
        # Estimate past popularity: average response rate (popularity) up to t-1
        mean_pjs = P_jt.iloc[:t].mean()
        top_items = mean_pjs.sort_values(ascending=False).index[:top_K]
        x_j = {j: 0 for j in candidates}
        for j in top_items:
            x_j[j] = 1/top_K
        # Normalize just in case
        total = sum(x_j.values())
        x_j = {j: v/total for j, v in x_j.items()}

    # Number of recommendations (visits) per item
    m_j = {j: int(n_visits * x_j[j]) for j in candidates}

    # --- Sample clicks per item using Binomial(Pjt, m_jt) ---
    c_j = {j: np.random.binomial(m_j[j], pjs[j]) for j in candidates}

    # --- Aggregate response rate for this interval
    clicks = sum(c_j.values())
    recs = sum(m_j.values())
    interval_response = clicks / recs if recs > 0 else 0

    overall_clicks += clicks
    overall_recs += recs

    # Print for debug/education
    print(f"--- Interval {t+1} ---")
    print(f"Serving fractions: {x_j}")
    print(f"Recommendation plan (m_j): {m_j}")
    print(f"Ground-truth responses/P_jt: {pjs}")
    print(f"Simulated clicks (c_j): {c_j}")
    print(f"Interval response rate: {interval_response:.4f}\n")

# --- Compute final metric ---
overall_response_rate = overall_clicks / overall_recs
print(f"=== Overall Response Rate Across {n_intervals} Intervals: {overall_response_rate:.4f} ===")